## 1.2 Optimizing Attention with FlashAttention-2
### **1.2.1 Benchmarking PyTorch Attention**
---

请编写一个脚本，在不同的尺度下对你实现的注意力机制进行基准测试。脚本应包含以下步骤：

*   **(a)** 将批大小（Batch Size）固定为 8，且不使用多头注意力机制（即：移除“头”维度，将其看作单头）。
*   **(b)** 遍历以下参数的笛卡尔积：头嵌入维度 $d_{model}$ 取值为 `[16, 32, 64, 128]`，序列长度（Sequence Length）取值为 `[256, 1024, 4096, 8192, 16384]`。
*   **(c)** 创建对应尺寸的随机输入张量 $Q, K, V$。
*   **(d)** 测量使用这些输入进行 100 次前向传播（Forward Passes）的时间。
*   **(e)** 在反向传播开始前，测量当前正在使用的显存量（Memory in use），并测量 100 次反向传播（Backward Passes）的时间。
*   **(f)** 确保进行预热（Warm up），并且在每次前向/反向传播之后调用 `torch.cuda.synchronize()`（以确保测速准确）。

**实验报告要求：**

汇报在这些配置下得到的耗时数据（如果报错则汇报显存溢出错误，即 Out-of-Memory, OOM）。
1.  在什么样的尺寸下会出现显存溢出（OOM）？
2.  针对你发现的出现 OOM 的最小配置之一，进行注意力机制显存占用的“账目计算”（你可以使用 Assignment 1 中关于 Transformer 显存占用的计算公式）。
3.  为了反向传播而保存（缓存）的显存量是如何随序列长度变化的？
4.  你会采取什么方法来消除这部分显存开销？

**交付物：**
一份包含耗时数据的表格、显存占用的计算推导过程，以及 1-2 段文字的回答。

---

In [ ]:
import os
import torch.nn.functional as F

def pytorch_naive_attention(q, k, v, mask=None):
    """
    实现朴素的 PyTorch Attention: softmax(Q @ K.T / sqrt(dk)) @ V
    
    参数:
        q: Query 张量，形状为 (batch_size, seq_len, d_model)
        k: Key 张量，形状为 (batch_size, seq_len, d_model)
        v: Value 张量，形状为 (batch_size, seq_len, d_model)
        mask: 可选的掩码张量 (例如 causal mask)
        
    返回:
        output: 注意力计算后的输出，形状为 (batch_size, seq_len, d_model)
    """
    #step1:calculate attentionScore
    attentionScore=q@k.transpose(-2,-1)

    #step2:normalize
    d_k=q.shape[-1]
    attentionScore=attentionScore/d_k**0.5

    #step3:mask
    if mask is not None:
        attentionScore=attentionScore.masked_fill(~mask,float('-inf'))

    #step4:softmax
    attentionScore=torch.softmax(attentionScore,dim=-1)

    #step5:calculate output
    output=attentionScore@v
    return output

In [ ]:
import time
import torch
import pandas as pd
import numpy as np

def benchmark_attention():
    batch_size = 8
    d_model_list = [16, 32, 64, 128]
    seq_len_list = [256, 1024, 4096, 8192, 16384]

    device = "cuda" if torch.cuda.is_available() else "cpu"
    results = []

    for d_model in d_model_list:
        for seq_len in seq_len_list:
            print(f"Testing: d_model={d_model}, seq_len={seq_len}...")
            
            # 清理显存缓存
            if device == "cuda":
                torch.cuda.empty_cache()
            
            try:
                # 构造数据，需要 requires_grad 来测试后向传播
                q = torch.randn(batch_size, seq_len, d_model, device=device, requires_grad=True)
                k = torch.randn(batch_size, seq_len, d_model, device=device, requires_grad=True)
                v = torch.randn(batch_size, seq_len, d_model, device=device, requires_grad=True)

                # --- 1. Warm up ---
                for _ in range(10):
                    _ = pytorch_naive_attention(q, k, v)

                # --- 2. Measure Forward Pass ---
                if device == "cuda": torch.cuda.synchronize()
                start_fwd = time.perf_counter()
                
                for _ in range(100):
                    out = pytorch_naive_attention(q, k, v)
                
                if device == "cuda": torch.cuda.synchronize()
                end_fwd = time.perf_counter()
                avg_fwd = (end_fwd - start_fwd) / 100 * 1000  # 转为 ms

                # --- 3. Measure Memory ---
                # 在后向传播开始前记录显存
                mem_usage = torch.cuda.memory_allocated(device) / (1024 ** 2) # MB

                # --- 4. Measure Backward Pass ---
                grad_output = torch.randn_like(out)
                if device == "cuda": torch.cuda.synchronize()
                start_bwd = time.perf_counter()
                
                for _ in range(100):
                    out.backward(grad_output, retain_graph=True)
                
                if device == "cuda": torch.cuda.synchronize()
                end_bwd = time.perf_counter()
                avg_bwd = (end_bwd - start_bwd) / 100 * 1000 # ms

                results.append({
                    "d_model": d_model,
                    "seq_len": seq_len,
                    "fwd_ms": f"{avg_fwd:.3f}",
                    "bwd_ms": f"{avg_bwd:.3f}",
                    "mem_mb": f"{mem_usage:.2f}"
                })

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"OOM at seq_len={seq_len}")
                    results.append({
                        "d_model": d_model,
                        "seq_len": seq_len,
                        "fwd_ms": "OOM",
                        "bwd_ms": "OOM",
                        "mem_mb": "OOM"
                    })
                else:
                    raise e

    # 打印结果表格 (作业 1.1.2 建议使用 pandas 打印表格)
    df = pd.DataFrame(results)
    print("\nBenchmark Results:")
    print(df.to_markdown()) # 需要 pip install tabulate

if __name__ == "__main__":
    benchmark_attention()


Benchmark Results:
|    |   d_model |   seq_len | fwd_ms   | bwd_ms   | mem_mb   |
|---:|----------:|----------:|:---------|:---------|:---------|
|  0 |        16 |       256 | 0.118    | 6.040    | 10.62    |
|  1 |        16 |      1024 | 1.241    | 2.177    | 50.38    |
|  2 |        16 |      4096 | 15.336   | 29.282   | 536.75   |
|  3 |        16 |      8192 | 64.601   | 120.728  | 2082.25  |
|  4 |        16 |     16384 | OOM      | OOM      | OOM      |
|  5 |        32 |       256 | 0.102    | 0.431    | 23.25    |
|  6 |        32 |      1024 | 1.241    | 2.184    | 52.50    |
|  7 |        32 |      4096 | 16.176   | 30.388   | 545.25   |
|  8 |        32 |      8192 | 67.603   | 128.251  | 2100.25  |
|  9 |        32 |     16384 | OOM      | OOM      | OOM      |
| 10 |        64 |       256 | 0.108    | 0.307    | 28.25    |
| 11 |        64 |      1024 | 1.306    | 2.512    | 56.75    |
| 12 |        64 |      4096 | 19.095   | 34.864   | 562.25   |
| 13 |        64 |      8192 | 81.289   | 152.397  | 2136.25  |
| 14 |        64 |     16384 | OOM      | OOM      | OOM      |
| 15 |       128 |       256 | 0.201    | 0.423    | 38.25    |
| 16 |       128 |      1024 | 1.807    | 3.657    | 65.25    |
| 17 |       128 |      4096 | 27.715   | 52.034   | 596.25   |
| 18 |       128 |      8192 | 127.961  | 238.462  | 2208.25  |
| 19 |       128 |     16384 | OOM      | OOM      | OOM      |

# 实验报告要求：

汇报在这些配置下得到的耗时数据（如果报错则汇报显存溢出错误，即 Out-of-Memory, OOM）。

## Q1：在什么样的尺寸下会出现显存溢出（OOM）？
针对你发现的出现 OOM 的最小配置之一，进行注意力机制显存占用的“账目计算”（你可以使用 Assignment 1 中关于 Transformer 显存占用的计算公式）。

**A:** 
当 `seq_len = 16384` 时，无论 `d_model` 大小是多少，都会发生显存溢出（OOM）。我们以出现 OOM 的最小配置之一（`batch_size = 8, d_model = 16, seq_len = 16384`，数据类型为 float32 即 4 bytes）为例，进行显存账目计算：

*   **输入张量 (Q, K, V Matrix)**：
    $$ \text{Memory}_{QKV} = \text{batch\_size} \times \text{d\_model} \times \text{seq\_len} \times 3 \text{ 个张量} \times 4 \text{ bytes} $$
    $$ \text{Memory}_{QKV} = 8 \times 16 \times 16384 \times 3 \times 4 \approx 25.16 \text{ MB} \approx 0.02 \text{ GB} $$

*   **前向传播中间激活 (Forward)**：
    在计算 `attentionScore = Q @ K.T` 时，会生成一个巨大的注意力矩阵，其大小为：
    $$ \text{Memory}_{Score} = \text{batch\_size} \times \text{seq\_len} \times \text{seq\_len} \times 4 \text{ bytes} $$
    $$ \text{Memory}_{Score} = 8 \times 16384 \times 16384 \times 4 = 8,589,934,592 \text{ bytes} = 8 \text{ GB} $$

*   **反向传播图保留 (Backward)**：
    为了能够计算梯度，PyTorch 的 Autograd 引擎必须在显存中**保留**前向传播生成的注意力概率矩阵（Softmax 的输出），这部分至少需要占用：
    $$ \text{Memory}_{Saved} = 8 \text{ GB} $$

*   **总计 (Total)**：
    $$ \text{Total} \approx \text{Memory}_{QKV} + \text{Memory}_{Score} (\text{前向计算中}) + \text{Memory}_{Saved} (\text{为反向保留}) > 16 \text{ GB} $$

**结论**：
计算该注意力机制的峰值显存需求远超 16GB。而 Kaggle 提供的单张 GPU T4 显存上限仅为 16GB（即使是 T4x2，在未开启分布式的单进程测试下也只能使用单卡的 16GB），因此必然触发 Out-of-Memory (OOM) 错误。这也是为什么我们需要引入 FlashAttention 来规避实例化 $O(N^2)$ 的注意力矩阵。
